### Datos faltantes: análisis inicial

Identificación de columnas con valores nulos y su posible relación con otras variables.

In [ ]:
import pandas as pd 
import numpy as np
from IPython.display import display
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu
import missingno as msno
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
#from functions.transformers import *
from functions.eda_functions import test_mcar, list_missing_values

In [ ]:
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
print(f'start dimensions: {df_train.shape}')
df_train.head()

In [ ]:
list_missing_values(df_train)

In [ ]:
#PoolQC: Calidad de la piscina.
print(f"-- PoolQC ---")
#Visualización de los datos faltantes, cantidad y porcentaje
print(df_train['PoolQC'].unique())
missing_poolqc = pd.DataFrame(df_train['PoolQC'].value_counts(dropna=False))
missing_poolqc['percentage(%)'] = np.round(missing_poolqc['count'] / df_train.shape[0] * 100, 2)
display(missing_poolqc)
#Visualización de los datos faltantes, area de piscina frente a calidad de piscina
res = df_train[df_train['PoolArea'] > 0]
display(res[['PoolArea', 'PoolQC']])
#Imputación de los datos faltantes
df_train.loc[:, 'PoolQC'] = df_train['PoolQC'].fillna('NA')
#Visualización de la relación entre PoolQC y PoolArea graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='PoolQC', y='PoolArea', data=df_train)
plt.title('Pool Quality vs Pool Area')
plt.show()
#PoolQC: Calidad de la piscina. depende de PoolArea. Si PoolArea es 0, PoolQC es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#MiscFeature: Característica miscelánea no cubierta en otras categorías.
print(f"-- MiscFeature ---")
#MiscVal: Valor monetario de la característica miscelánea.
#Visualización de los datos faltantes, cantidad y porcentaje
missing_miscFeature = pd.DataFrame(df_train['MiscFeature'].value_counts(dropna=False))
missing_miscFeature['percentage(%)'] = np.round(missing_miscFeature['count'] / df_train.shape[0] * 100, 2)
display(missing_miscFeature)
#visualizar las filas con datos faltantes en MiscFeature y su relación con MiscVal
match_miscval = df_train[df_train['MiscVal'] > 0]
display(match_miscval[['MiscFeature', 'MiscVal']])
#Imputación de los datos faltantes
df_train.loc[:, 'MiscFeature'] = df_train['MiscFeature'].fillna('NA')
#Visualizacion de la relación entre MiscFeature y MiscVal graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='MiscFeature', y='MiscVal', data=df_train)
plt.title('Misc Feature vs Misc Val')
plt.show()
#MiscFeature: Característica miscelánea no cubierta en otras categorías. Depende de MiscVal. Si MiscVal es 0, MiscFeature es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#Alley: Tipo de acceso por callejón a la propiedad.
print(f"-- Alley ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_alley = pd.DataFrame(df_train['Alley'].value_counts(dropna=False))
missing_alley['percentage(%)'] = np.round(missing_alley['count'] / df_train.shape[0] * 100, 2)
display(missing_alley)
#visualizar las filas con datos faltantes en Alley y su relación con LotFrontage
test_mcar(df_train, 'Alley')
df_train.loc[:, 'Alley'] = df_train['Alley'].fillna('NA')
print('group lotArea by Alley')
display(df_train.groupby(['Alley'])['LotArea'].agg(['count', 'mean']).reset_index())
print('group SalePrice by Alley')
display(df_train.groupby(['Alley'])['SalePrice'].agg(['count', 'mean']).reset_index())
print('group LotFrontage by Alley')
display(df_train.groupby(['Alley'])['LotFrontage'].agg(['count', 'mean']).reset_index())
#Imputación de los datos faltantes con valor NA, que indica que no tiene callejon.
#Se analizo una posible MCAR, pero no se encontró evidencia suficiente.
#Se determino en la comparacion que tiene un manejo diferente los grupos donde Alley es NA y donde no lo es.
#Se define como MAR, ya que las casas sin callejon tienen a ser mas caras

#En las agrupaciones con LotArea, SalePrice y LotFrontage se observa que las propiedades sin Alley (NA) tienen promedios más altos en comparación con las que tienen Alley.
#Esto sugiere que la ausencia de Alley está asociada con propiedades más grandes y valiosas.
#MAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#Fence: Tipo de cerca alrededor de la propiedad.
print(f"-- Fence ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_fence = pd.DataFrame(df_train['Fence'].value_counts(dropna=False))
missing_fence['percentage(%)'] = np.round(missing_fence['count'] / df_train.shape[0] * 100, 2)
display(missing_fence)
test_mcar(df_train, 'Fence')
df_train.loc[:, 'Fence'] = df_train['Fence'].fillna('NA')
print('group LotArea by Fence')
display(df_train.groupby(['Fence'])['LotArea'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False))
print('group LotFrontage by Fence')
display(df_train.groupby(['Fence'])['LotFrontage'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False))
print('group SalePrice by Fence')
display(df_train.groupby(['Fence'])['SalePrice'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False))
#Imputación de los datos faltantes con valor NA, que indica que no tiene cerca.
#Se analizo una posible MCAR, pero no se encontró evidencia suficiente.
#Se determino en la comparacion que tiene un manejo diferente los grupos donde Fence es NA y donde no lo es.
#Se define como MAR, ya que las casas sin cerca tienen a ser mas caras

In [ ]:
#MasVnrType: Tipo de revestimiento de mampostería.
print(f"-- MasVnrType ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_masvnrtype = pd.DataFrame(df_train['MasVnrType'].value_counts(dropna=False))
missing_masvnrtype['percentage(%)'] = np.round(missing_masvnrtype['count'] / df_train.shape[0] * 100, 2)
display(missing_masvnrtype)
#Se da por hecho que si MasVnrArea es 0, entonces MasVnrType es NA
condition_impute = (df_train['MasVnrType'].isnull()) & (df_train['MasVnrArea'] == 0)
df_train.loc[condition_impute, 'MasVnrType'] = df_train['MasVnrType'].fillna('NA')
#Basado en Foundation: cimientos se imputa por el valor similar CBlock y Pconc
condition_impute2 = (df_train['MasVnrType'].isnull()) & ((df_train['Foundation'] == 'CBlock') | (df_train['Foundation'] == 'PConc'))
df_train.loc[condition_impute2, 'MasVnrType'] = df_train['MasVnrType'].fillna('CBlock')
df_train['MasVnrType'].unique()

In [ ]:
#FireplaceQu: Calidad de la chimenea.
print(f"-- FireplaceQu ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_fireplacequ = pd.DataFrame(df_train['FireplaceQu'].value_counts(dropna=False))
missing_fireplacequ['percentage(%)'] = np.round(missing_fireplacequ['count'] / df_train.shape[0] * 100, 2)
display(missing_fireplacequ)
#Se da por hecho que si Fireplaces es 0, entonces FireplaceQu es NA
condition_fireplacequ = (df_train['FireplaceQu'].isnull()) & (df_train['Fireplaces'] == 0)
df_train.loc[condition_fireplacequ, 'FireplaceQu'] = df_train['FireplaceQu'].fillna('NA')

In [ ]:
#LotFrontage: Metros lineales de la calle conectada a la propiedad.
print(f"-- LotFrontage ---")
df_train['LotFrontage_null'] = np.where(df_train['LotFrontage'].isnull(), 1, 0)
df_train['LotFrontage_null'].value_counts()
df_train.groupby(['LotFrontage_null', 'MSZoning'])['LotArea'].agg(['count', 'mean', 'median']).reset_index().sort_values(ascending=True, by='mean')